# CO 370 — Sensitivity Analysis

Three experiments on the meal planning MIP:
1. Lambda sweep (waste penalty 0 to 2)
2. Repetition limits (max uses per lunch/dinner, 1 to 5)
3. Caloric profiles (sedentary female / default / active male)

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import gurobipy as gp
from gurobipy import GRB

from model import load_data, build_params, build_model, NUT_BOUNDS_RELAXED, DAYS

os.makedirs("results", exist_ok=True)
QUIET = {"OutputFlag": 0, "LogToConsole": 0}

In [ ]:
def solve_instance(ingredients, meal_ids, B, L, D, p, s, r, a,
                   lam=0.5, nut_bounds=None,
                   rep_b=3, rep_l=2, rep_d=2,
                   cal_lb=1800, cal_ub=2500):
    """Build and solve one MIP instance."""
    if nut_bounds is None:
        nut_bounds = NUT_BOUNDS_RELAXED

    bounds = dict(nut_bounds)
    bounds["calories"] = (cal_lb, cal_ub)

    m, x, y, w = build_model(
        ingredients, meal_ids, B, L, D, p, s, r, a,
        lam=lam, nut_bounds=bounds
    )

    for constr in m.getConstrs():
        name = constr.ConstrName
        if name.startswith("C4_"): constr.RHS = rep_b
        elif name.startswith("C5_"): constr.RHS = rep_l
        elif name.startswith("C6_"): constr.RHS = rep_d

    for k, v in QUIET.items():
        m.setParam(k, v)
    m.optimize()

    if m.Status != GRB.OPTIMAL:
        return {"status": m.Status}

    grocery = sum(p[j] * round(y[j].X) for j in ingredients)
    waste_raw = sum((p[j] / s[j]) * w[j].X for j in ingredients)
    return {
        "status": GRB.OPTIMAL, "obj": m.ObjVal,
        "grocery_cost": grocery, "waste_cost_raw": waste_raw,
        "waste_penalty": lam * waste_raw,
    }

In [ ]:
prices, meals, meal_ings, meal_nuts = load_data()
ingredients, meal_ids, B, L, D, p, s, r, a = build_params(
    prices, meals, meal_ings, meal_nuts
)
print(f"{len(meal_ids)} meals | {len(ingredients)} ingredients")

## Experiment 1 — Lambda Sweep

Vary `lambda` from 0 to 2. At lambda=0 the model only minimizes grocery cost; higher values penalize waste more.

In [ ]:
lambdas = [0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
rows = []
for lam in lambdas:
    res = solve_instance(ingredients, meal_ids, B, L, D, p, s, r, a, lam=lam)
    if res["status"] == GRB.OPTIMAL:
        rows.append({
            "lambda": lam, "grocery_cost": res["grocery_cost"],
            "waste_cost_raw": res["waste_cost_raw"],
            "waste_penalty": res["waste_penalty"], "total_obj": res["obj"],
        })

df_lam = pd.DataFrame(rows)
df_lam.to_csv("results/lambda_sensitivity.csv", index=False)
df_lam

## Experiment 2 — Repetition Limits

Vary the max number of times each lunch/dinner can repeat during the week (1 to 5).

In [ ]:
rows = []
for rep in [1, 2, 3, 4, 5]:
    res = solve_instance(
        ingredients, meal_ids, B, L, D, p, s, r, a,
        lam=0.5, rep_b=3, rep_l=rep, rep_d=rep
    )
    if res["status"] == GRB.OPTIMAL:
        rows.append({
            "rep_limit": rep, "grocery_cost": res["grocery_cost"],
            "waste_cost_raw": res["waste_cost_raw"],
        })
    else:
        rows.append({"rep_limit": rep, "grocery_cost": None, "waste_cost_raw": None})

df_rep = pd.DataFrame(rows).dropna()
df_rep.to_csv("results/repetition_sensitivity.csv", index=False)
df_rep

## Experiment 3 — Caloric Profiles

Compare cost across three calorie ranges representing different user profiles.

In [ ]:
profiles = [
    ("Sedentary Female (1600-2200)", 1600, 2200),
    ("Default (1800-2500)",          1800, 2500),
    ("Active Male (2000-2800)",      2000, 2800),
]
rows = []
for label, lb, ub in profiles:
    res = solve_instance(
        ingredients, meal_ids, B, L, D, p, s, r, a,
        lam=0.5, cal_lb=lb, cal_ub=ub
    )
    status = "optimal" if res["status"] == GRB.OPTIMAL else "infeasible"
    rows.append({
        "profile": label, "cal_lb": lb, "cal_ub": ub,
        "grocery_cost": res.get("grocery_cost"),
        "waste_cost_raw": res.get("waste_cost_raw"),
        "status": status,
    })

df_cal = pd.DataFrame(rows)
df_cal.to_csv("results/caloric_profiles.csv", index=False)
df_cal